# Credit Card Fraud Detection
## Oasis Infobyte Data Analytics Internship — Level 2, Task 3

**Author:** Gangula Hemanth Varma  
**Dataset:** ULB Credit Card Fraud Detection (Kaggle)  
**Date:** 2026  

---

## Project Objective
Build a machine-learning pipeline to detect fraudulent credit card transactions 
from a heavily imbalanced real-world dataset. The project covers:
- Full exploratory data analysis
- Class imbalance handling with SMOTE
- Logistic Regression and Random Forest classifiers
- Comprehensive evaluation (Precision, Recall, F1, ROC-AUC)
- Business interpretation of results
- Scalability discussion for production deployment


## 0. Imports & Configuration


In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings('ignore')

# Project paths
BASE_DIR  = os.path.dirname(os.path.abspath('.'))
DATA_PATH = '../data/creditcard.csv'
MODEL_DIR = '../models'
PLOT_DIR  = '../models/plots'

# Colour palette
FRAUD_COL = '#E74C3C'
LEGIT_COL = '#2ECC71'
LR_COL    = '#3498DB'
RF_COL    = '#F39C12'

plt.rcParams.update({'figure.facecolor':'#F8F9FA','axes.facecolor':'#F8F9FA',
                     'axes.grid':True,'grid.color':'#DEE2E6',
                     'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
print('Setup complete')


## 1. Dataset Loading & Inspection

The Credit Card Fraud Detection dataset was collected by the ULB Machine Learning Group.
It contains 284,807 transactions made by European cardholders over two days in September 2013.
Features V1-V28 are PCA-transformed for privacy. Only `Amount` and `Time` retain their original scale.


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head(3)


In [ ]:
# Data types and missing values
print('=== Data Types ===')
print(df.dtypes.value_counts())
print()
print(f'Total missing values: {df.isnull().sum().sum()}')
print('No missing values — dataset is clean.')


In [ ]:
# Summary statistics for Amount and Time
df[['Time','Amount','Class']].describe().round(2)


## 2. Class Distribution & Imbalance Analysis

### The Severe Class Imbalance Problem

This dataset has an **extreme class imbalance**: only 0.17% of transactions are fraudulent.
For every single fraud, there are 577 legitimate transactions.

**Why this matters for model evaluation:**
A naive classifier that predicts 'Legitimate' for *every* transaction achieves **99.83% accuracy**.
This looks impressive but is completely useless — it catches zero frauds.
Standard accuracy is therefore a misleading metric for this problem.
We must focus on **Precision, Recall, F1-score, and ROC-AUC** instead.


In [ ]:
vc = df['Class'].value_counts()
n  = len(df)
print('Class Distribution (REAL)')
print(f'  Legitimate (0): {vc[0]:,}  ({vc[0]/n*100:.4f}%)')
print(f'  Fraudulent (1): {vc[1]:,}  ({vc[1]/n*100:.4f}%)')
print(f'  Imbalance ratio: 1:{int(vc[0]/vc[1])}')
print()
print(f'  A dummy classifier accuracy: {vc[0]/n*100:.2f}% (useless baseline)')


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,5))
fig.suptitle('Credit Card Fraud — Class Distribution', fontsize=14, fontweight='bold')

axes[0].bar(['Legitimate (0)','Fraudulent (1)'],[vc[0],vc[1]],
            color=[LEGIT_COL,FRAUD_COL],edgecolor='white',width=0.5)
axes[0].set_title('Transaction Count')
axes[0].set_ylabel('Count')
for i,(v) in enumerate([vc[0],vc[1]]):
    axes[0].text(i, v+1000, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie([vc[0],vc[1]],labels=['Legitimate\n99.83%','Fraudulent\n0.17%'],
            colors=[LEGIT_COL,FRAUD_COL],explode=[0,0.12],autopct='%1.2f%%',
            startangle=140,wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Class Proportion (1:577 imbalance)')
plt.tight_layout()
plt.savefig('../screenshots/nb_01_class_distribution.png',dpi=120,bbox_inches='tight')
plt.show()
print('Interpretation: The extreme imbalance means accuracy alone is useless.')
print('We must use Precision, Recall, F1, and AUC.')


## 3. Exploratory Data Analysis (EDA)

### 3.1 Transaction Amount Analysis


In [ ]:
fraud = df[df['Class']==1]['Amount']
legit = df[df['Class']==0]['Amount']

print('=== Transaction Amount Statistics ===')
print(f'FRAUD  mean: ${fraud.mean():.2f}  median: ${fraud.median():.2f}  max: ${fraud.max():.2f}')
print(f'LEGIT  mean: ${legit.mean():.2f}  median: ${legit.median():.2f}  max: ${legit.max():.2f}')
print()
print('Key insight: Fraud median ($9.25) << Legit median ($22.00)')
print('Many fraudulent transactions are small test amounts, not large withdrawals.')


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Transaction Amount Distribution by Class', fontsize=13, fontweight='bold')

axes[0].hist(legit,bins=80,color=LEGIT_COL,alpha=0.7,density=True,
             label=f'Legitimate (n={len(legit):,})')
axes[0].hist(fraud,bins=40,color=FRAUD_COL,alpha=0.85,density=True,
             label=f'Fraudulent (n={len(fraud):,})')
axes[0].set_yscale('log')
axes[0].set_xlabel('Amount (EUR)')
axes[0].set_ylabel('Density (log)')
axes[0].set_title('Full Range (log y-axis)')
axes[0].legend()

axes[1].hist(legit[legit<=500],bins=60,color=LEGIT_COL,alpha=0.7,density=True,label='Legitimate')
axes[1].hist(fraud[fraud<=500],bins=40,color=FRAUD_COL,alpha=0.85,density=True,label='Fraudulent')
axes[1].set_xlabel('Amount (EUR)')
axes[1].set_ylabel('Density')
axes[1].set_title('Zoomed: up to EUR 500')
axes[1].legend()
plt.tight_layout()
plt.show()
print('Interpretation: Fraudulent amounts cluster near zero — typical card-testing behaviour.')


### 3.2 Time-of-Day Analysis


In [ ]:
df2 = df.copy()
df2['Hour'] = (df2['Time'] // 3600 % 24).astype(int)
fraud_hr = df2[df2['Class']==1].groupby('Hour').size()
legit_hr = df2[df2['Class']==0].groupby('Hour').size()
rate_hr  = df2.groupby('Hour')['Class'].mean()*100

fig, axes = plt.subplots(2,1,figsize=(14,8))
fig.suptitle('Fraud Patterns by Hour of Day (48-hour window mapped 0-23h)',fontsize=13,fontweight='bold')

hours = range(24)
x = np.array(list(hours))
ax,ax2 = axes[0], axes[0].twinx()
ax.bar(x-0.2,[legit_hr.get(h,0) for h in hours],0.4,color=LEGIT_COL,alpha=0.8,label='Legitimate')
ax2.bar(x+0.2,[fraud_hr.get(h,0) for h in hours],0.4,color=FRAUD_COL,alpha=0.85,label='Fraudulent')
ax.set_ylabel('Legitimate Count',color=LEGIT_COL)
ax2.set_ylabel('Fraudulent Count',color=FRAUD_COL)
ax.set_title('Transaction Volume by Hour')
ax.set_xticks(list(hours))
p1=mpatches.Patch(color=LEGIT_COL,label='Legitimate')
p2=mpatches.Patch(color=FRAUD_COL,label='Fraudulent')
ax.legend(handles=[p1,p2])

rate_vals=[float(rate_hr.get(h,0)) for h in hours]
axes[1].bar(list(hours),rate_vals,color=FRAUD_COL,alpha=0.8,edgecolor='white')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate (%) by Hour')
axes[1].set_xticks(list(hours))
peak = int(np.argmax(rate_vals))
axes[1].annotate(f'Peak: h{peak}',xy=(peak,rate_vals[peak]),
    xytext=(peak+1.5,rate_vals[peak]+0.01),arrowprops=dict(arrowstyle='->'),fontsize=9)
plt.tight_layout()
plt.show()
print(f'Highest fraud rate at hour {peak}: {rate_vals[peak]:.3f}%')


### 3.3 Feature Correlations with Fraud


In [ ]:
corr = df.corr()['Class'].drop('Class').sort_values(key=abs, ascending=False)
top20 = corr.head(20)

fig, ax = plt.subplots(figsize=(10,7))
colors = [FRAUD_COL if v>0 else LEGIT_COL for v in top20.values]
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0,color='black',lw=0.8)
ax.set_xlabel('Pearson Correlation with Class')
ax.set_title('Top 20 Features Correlated with Fraud\n(Red=positive, Green=negative)')
plt.tight_layout()
plt.show()
print('Top 3 most fraud-correlated features:')
print(corr.head(3).to_string())


### 3.4 Top PCA Feature Distributions


In [ ]:
top6 = corr.abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2,3,figsize=(15,8))
fig.suptitle('Top 6 Features: Fraud vs Legitimate Distributions', fontsize=13, fontweight='bold')
for i,feat in enumerate(top6):
    ax=axes.flatten()[i]
    ax.hist(df[df['Class']==0][feat],bins=60,color=LEGIT_COL,alpha=0.6,density=True,label='Legit')
    ax.hist(df[df['Class']==1][feat],bins=40,color=FRAUD_COL,alpha=0.75,density=True,label='Fraud')
    c=float(df.corr()['Class'][feat])
    ax.set_title(f'{feat}  (r={c:.3f})')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('These features show the clearest separation between fraud and legitimate transactions.')


## 4. Data Preprocessing

### 4.1 Feature Engineering
We derive `Hour` (0-23) from `Time` to capture time-of-day patterns, then drop raw `Time`.


In [ ]:
df['Hour'] = (df['Time'] // 3600 % 24).astype(int)
feature_cols = [c for c in df.columns if c not in ('Class','Time')]
X = df[feature_cols].copy()
y = df['Class'].copy()
print(f'Features: {len(feature_cols)}')
print(feature_cols[:5], '...')


### 4.2 Stratified Train/Test Split

We use **stratified** splitting to preserve the original 1:577 fraud ratio in both the training
and test sets. The test set is **never touched by SMOTE** — it represents reality.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

print(f'Train: {len(X_train):,} samples | Fraud: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Test:  {len(X_test):,} samples | Fraud: {y_test.sum():,} ({y_test.mean()*100:.3f}%)')
print()
print('Both sets maintain the original 0.17% fraud rate — no leakage.')


### 4.3 Feature Scaling

The scaler is **fit only on training data** then applied to both sets.
This prevents test-set statistics from influencing the scaler — a form of data leakage prevention.


In [ ]:
scaler = joblib.load('../models/scaler.pkl')
cols_to_scale = ['Amount','Hour']
X_train_s = X_train.copy()
X_test_s  = X_test.copy()
X_train_s[cols_to_scale] = scaler.transform(X_train_s[cols_to_scale])
X_test_s[cols_to_scale]  = scaler.transform(X_test_s[cols_to_scale])
print('Amount and Hour scaled (fit on train only).')
print(f'Amount stats after scaling — mean: {X_train_s["Amount"].mean():.3f}, std: {X_train_s["Amount"].std():.3f}')


### 4.4 SMOTE — Synthetic Minority Oversampling

#### Why SMOTE?
With a 1:577 ratio, any standard classifier will simply predict 'Legitimate' for every
transaction and achieve 99.83% accuracy. SMOTE creates synthetic fraud samples in the
*feature space* (interpolating between real fraud neighbours) to rebalance the training set.

#### Why ONLY on training data?
The test set must mirror the real-world distribution. If we applied SMOTE to the test set:
- Recall would be artificially inflated (more fraud samples to detect)
- Metrics would be optimistic and non-representative
- The model would appear better than it actually is in production

**Rule: Fit SMOTE on training data. Evaluate on raw, unmodified test data.**


In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)

before = y_train.value_counts()
after  = pd.Series(y_train_sm).value_counts()
print('Before SMOTE (training set):')
print(f'  Legitimate: {before[0]:,} | Fraudulent: {before[1]:,}')
print()
print('After SMOTE (training set only):')
print(f'  Legitimate: {after[0]:,} | Fraudulent: {after[1]:,}')
print(f'  Total: {len(X_train_sm):,}')
print()
print('Test set unchanged:', y_test.value_counts().to_dict())


## 5. Model Training

Both models are loaded from saved artifacts (trained on the full SMOTE-balanced set).
This notebook re-evaluates them on the same held-out test set for transparency.


In [ ]:
lr_model = joblib.load('../models/logistic_regression.pkl')
rf_model = joblib.load('../models/random_forest.pkl')
print('Models loaded successfully.')
print(f'LR params: {lr_model.get_params()}')
print(f'RF n_estimators: {rf_model.n_estimators}, max_depth: {rf_model.max_depth}')


## 6. Model Evaluation

### Why Not Just Use Accuracy?

With 99.83% legitimate transactions, a model predicting everything as 'legitimate' scores
**99.83% accuracy** but catches **0 frauds**. This is the **accuracy paradox**.

For fraud detection we need:
- **Precision**: Of all flagged frauds, how many are real? (avoids annoying customers)
- **Recall**: Of all real frauds, how many did we catch? (avoids missing frauds)
- **F1**: Harmonic mean — balanced trade-off
- **ROC-AUC**: Overall discrimination ability across all thresholds


In [ ]:
def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    cm = confusion_matrix(y_test, y_pred)
    tn,fp,fn,tp = cm.ravel()
    print(f'--- {name} ---')
    print(f'  Precision : {precision_score(y_test,y_pred):.4f}')
    print(f'  Recall    : {recall_score(y_test,y_pred):.4f}')
    print(f'  F1 Score  : {f1_score(y_test,y_pred):.4f}')
    print(f'  ROC-AUC   : {roc_auc_score(y_test,y_prob):.4f}')
    print(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp}')
    print()
    return y_pred, y_prob

lr_pred, lr_prob = evaluate('Logistic Regression', lr_model, X_test_s, y_test)
rf_pred, rf_prob = evaluate('Random Forest',       rf_model, X_test_s, y_test)


### 6.1 Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Confusion Matrices — Real Test Set Results', fontsize=13, fontweight='bold')

for ax, pred, name, col in [
    (axes[0], lr_pred, 'Logistic Regression', LR_COL),
    (axes[1], rf_pred, 'Random Forest',       RF_COL)]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred: Legit','Pred: Fraud'],
                yticklabels=['Act: Legit','Act: Fraud'],
                ax=ax, linewidths=2, linecolor='white', cbar=False,
                annot_kws={'size':14,'weight':'bold'})
    tn,fp,fn,tp=cm.ravel()
    ax.set_title(f'{name}\nTP={tp}  TN={tn}  FP={fp}  FN={fn}')
plt.tight_layout()
plt.show()


### 6.2 ROC Curves


In [ ]:
lr_fpr,lr_tpr,_ = roc_curve(y_test,lr_prob)
rf_fpr,rf_tpr,_ = roc_curve(y_test,rf_prob)
lr_auc = auc(lr_fpr,lr_tpr)
rf_auc = auc(rf_fpr,rf_tpr)

fig,ax = plt.subplots(figsize=(8,7))
ax.plot(lr_fpr,lr_tpr,color=LR_COL,lw=2,label=f'Logistic Regression (AUC={lr_auc:.4f})')
ax.plot(rf_fpr,rf_tpr,color=RF_COL,lw=2,label=f'Random Forest (AUC={rf_auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random (AUC=0.5000)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 20% Held-Out Test Set')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
print(f'Both models significantly outperform random. LR AUC={lr_auc:.4f}, RF AUC={rf_auc:.4f}')


### 6.3 Precision-Recall Curves


In [ ]:
lr_p,lr_r,_ = precision_recall_curve(y_test,lr_prob)
rf_p,rf_r,_ = precision_recall_curve(y_test,rf_prob)
lr_ap = average_precision_score(y_test,lr_prob)
rf_ap = average_precision_score(y_test,rf_prob)
baseline = y_test.mean()

fig,ax = plt.subplots(figsize=(8,7))
ax.plot(lr_r,lr_p,color=LR_COL,lw=2,label=f'Logistic Regression (AP={lr_ap:.4f})')
ax.plot(rf_r,rf_p,color=RF_COL,lw=2,label=f'Random Forest (AP={rf_ap:.4f})')
ax.axhline(baseline,color='gray',ls='--',lw=1,label=f'Baseline (AP={baseline:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves\n(crucial for imbalanced datasets)')
ax.legend()
plt.tight_layout()
plt.show()
print('RF has much higher Average Precision — better at precise fraud flagging.')


### 6.4 Model Comparison Table


In [ ]:
with open('../models/model_metrics.json') as f:
    metrics = json.load(f)

lr_m = metrics['logistic_regression']
rf_m = metrics['random_forest']

comparison = pd.DataFrame({
    'Metric':   ['Precision','Recall','F1 Score','ROC-AUC','Avg Precision','Training Time (s)'],
    'Logistic Regression': [
        lr_m['precision'], lr_m['recall'], lr_m['f1_score'],
        lr_m['roc_auc'], lr_m['avg_precision'], lr_m['training_time_seconds']],
    'Random Forest': [
        rf_m['precision'], rf_m['recall'], rf_m['f1_score'],
        rf_m['roc_auc'], rf_m['avg_precision'], rf_m['training_time_seconds']],
})
comparison.set_index('Metric', inplace=True)
comparison.style.highlight_max(axis=1, color='#d4edda').format('{:.4f}')


## 7. Business Interpretation

### 7.1 Precision vs Recall Trade-off in Fraud Detection

| Metric | What it measures | Business meaning |
|--------|-----------------|------------------|
| **Precision** | Of all flagged transactions, % truly fraudulent | How often do we block a real customer by mistake? |
| **Recall** | Of all real frauds, % we caught | How many frauds did we miss? |
| **False Positive** | Legitimate tx flagged as fraud | Customer frustration, blocked cards, support costs |
| **False Negative** | Fraud tx missed | Financial loss, regulatory exposure, reputational damage |

### 7.2 Which Metric Is More Important?

The answer depends on the business context:

**High-Recall priority (catch every fraud):**
- Used in: card-present transactions, high-value wire transfers, new-account detection
- Acceptable to block some legitimate transactions (bank absorbs fraud liability)
- Logistic Regression (recall=0.9184) is better here but generates 1,526 FP per test batch

**High-Precision priority (minimise false alarms):**
- Used in: low-friction e-commerce checkout, travel cards, VIP customer accounts
- Customers have low tolerance for blocked transactions
- Random Forest (precision=0.8247) is far superior here with only 17 FP

### 7.3 Confusion Matrix in Business Terms (Random Forest, 56,962 test transactions)

| | Predicted: Legitimate | Predicted: Fraud |
|-|----------------------|-----------------|
| **Actual: Legitimate** | 56,847 correctly approved (TN) | 17 customers unnecessarily blocked (FP) |
| **Actual: Fraud** | 18 fraudulent transactions missed (FN) | 80 frauds correctly caught (TP) |

**Impact:**
- 17 false positives → 17 frustrated customers needing support calls
- 18 false negatives → up to 18 × average fraud amount = up to 18 × $122 = ~$2,196 undetected fraud per batch

### 7.4 Recommended Model: Random Forest

Random Forest is the preferred model for production deployment because:
- **F1 score of 0.8205** vs 0.1050 for LR — far better overall balance
- Only **17 false positives** vs 1,526 for LR — drastically fewer legitimate customers blocked
- **Comparable ROC-AUC** (0.9683 vs 0.9706) — similar overall discrimination
- More robust to non-linear feature interactions in PCA-transformed features

Logistic Regression remains valuable as a **high-sensitivity first-pass filter** in a
two-stage system: LR flags broadly, then RF validates — combining high recall with high precision.


## 8. Feature Analysis

### 8.1 Logistic Regression Coefficients

In logistic regression, a positive coefficient increases the log-odds of fraud;
a negative coefficient decreases them. Magnitude indicates importance.

**Note:** V1-V28 are PCA components — their original business meaning is confidential.
We can only state which transformed dimensions discriminate fraud from legitimate transactions.


In [ ]:
coef_df = pd.read_json('../models/lr_coefficients.json')
top15 = coef_df.head(15)

fig, ax = plt.subplots(figsize=(10,6))
colors = [FRAUD_COL if v>0 else LEGIT_COL for v in top15['coefficient']]
ax.barh(top15['feature'][::-1], top15['coefficient'][::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Logistic Regression — Top 15 Features by |Coefficient|')
plt.tight_layout()
plt.show()
print('Top 5 fraud-increasing features (positive coefficients):')
print(coef_df[coef_df['coefficient']>0].head(5)[['feature','coefficient']].to_string(index=False))


### 8.2 Random Forest Feature Importances


In [ ]:
fi_df = pd.read_json('../models/rf_feature_importance.json')
top15 = fi_df.head(15)

fig, ax = plt.subplots(figsize=(10,6))
colors = plt.cm.YlOrRd(np.linspace(0.4,0.9,len(top15)))
ax.barh(top15['feature'][::-1], top15['importance'][::-1], color=colors, edgecolor='white')
ax.set_xlabel('Gini Importance')
ax.set_title('Random Forest — Top 15 Feature Importances')
plt.tight_layout()
plt.show()
print('Top 5 most important features (RF):')
print(fi_df.head(5)[['feature','importance']].to_string(index=False))
print()
print('Interpretation: High RF importance on V14, V17, V12, V10 etc.')
print('These PCA dimensions carry the most discriminative signal for fraud detection.')
print('Amount appears in both models — transaction size has genuine predictive power.')


## 9. Scalability Discussion — Production at ~1M Transactions/Hour

Scaling this proof-of-concept to a production fraud-detection system serving ~1 million
transactions per hour (~278 tx/sec) requires a fundamentally different architecture.

### 9.1 Current Demo Architecture
```
CSV file  →  Python script  →  sklearn model  →  JSON output
```
This works for analysis but cannot handle real-time volume.

### 9.2 Target Production Architecture
```
Transactions  →  API Gateway  →  Feature Service  →  Model Server  →  Decision
                                        ↓                    ↓
                                  Feature Store        Model Registry
                                        ↓                    ↓
                                  Stream Processor    Monitoring/Drift
```

### 9.3 Key Architectural Components

**1. Ingestion Layer**
- Apache Kafka or AWS Kinesis for event streaming at 278+ tx/sec
- Partitioning by card hash ensures ordering per customer

**2. Feature Engineering Service**
- Low-latency feature store (Redis, Feast, or Tecton)
- Pre-computed velocity features: tx count in last 1/5/60 minutes
- Avoids re-computing PCA in real time (use pre-stored transformed vectors)

**3. Model Serving**
- Deploy RF model via ONNX Runtime or Triton Inference Server
- Target latency: <10ms per prediction (p99)
- Horizontal scaling: multiple model replicas behind a load balancer
- Container orchestration: Kubernetes with HPA (auto-scaling on CPU/req rate)

**4. Batching vs Streaming**
- Real-time: Use streaming (Kafka + Faust/Spark Streaming) for card-present
- Batch: Nightly re-scoring of suspicious accounts is acceptable for low-risk review
- Micro-batching (50-100 tx batches, <100ms) balances throughput and latency

**5. Model Retraining Pipeline**
- Fraud patterns drift over time (adversarial adaptation)
- Automated weekly retraining on new labelled data
- A/B shadow deployment: new model serves alongside live model, metrics compared
- Champion/Challenger pattern: promote new model only if AUC improves > 0.005

**6. Monitoring & Alerting**
- Data drift detection: PSI (Population Stability Index) on feature distributions
- Concept drift: monitor precision/recall on confirmed fraud labels
- Latency SLA alerts: p99 > 50ms triggers auto-scaling
- False positive rate alerts: spike in FP → possible model degradation

**7. Compliance & Explainability**
- PSD2 / GDPR require explainable decisions affecting customers
- SHAP values computed per prediction for dispute resolution
- All predictions logged to immutable audit trail (S3 + Athena)

### 9.4 Infrastructure Estimate (1M tx/hour)

| Component | Technology | Instances |
|-----------|-----------|-----------|
| API Gateway | AWS API GW / Kong | Auto-scaled |
| Stream Processing | Kafka (6 brokers) | 6 × m5.xlarge |
| Feature Store | Redis Cluster | 3 × r6g.large |
| Model Serving | Triton (ONNX) | 4-8 × c5.2xlarge |
| Database | PostgreSQL RDS | 1 primary + 2 read replicas |
| Monitoring | Prometheus + Grafana | 1 × t3.medium |

### 9.5 Cost vs Accuracy Trade-off
- Increasing model complexity (XGBoost, deep learning) can improve recall by 1-2%
- But at 278 tx/sec, even a 5ms latency increase has significant infrastructure cost
- Random Forest with ONNX serialisation achieves <5ms latency — a good production balance


## 10. Conclusion

This project built a complete, end-to-end fraud detection pipeline on the real ULB dataset.

### Key Findings

| Finding | Detail |
|---------|--------|
| Dataset | 284,807 transactions, 492 fraud (0.17%), zero missing values |
| Imbalance | 1:577 ratio — standard accuracy is completely misleading |
| SMOTE | Rebalanced training to 227,451:227,451 — no test leakage |
| Best model | Random Forest: Precision=0.8247, Recall=0.8163, F1=0.8205, AUC=0.9683 |
| LR trade-off | High recall (0.9184) but very low precision (0.0557) — 1,526 false alarms |
| Key features | V14, V17, V12, V10 are the most discriminative PCA dimensions |
| Fraud amounts | Cluster near zero — typical card-testing behaviour |

### Limitations
- V1-V28 are PCA-transformed: original feature meanings cannot be recovered
- Dataset covers only 2 days: temporal generalisation is unknown
- No hyperparameter optimisation performed (GridSearchCV would further improve RF)
- Production deployment requires additional security, compliance, and monitoring layers

### Next Steps
- XGBoost / LightGBM comparison
- Threshold tuning for specific FP/FN business cost ratios
- SHAP explainability for regulatory compliance
- Online learning for real-time model adaptation
